In [31]:
import pandas as pd
import statsmodels.api as sm

In [32]:
def backward_elimination(y, X, significance_level=0.10):
    """
    Iteratively removes variables with p-values > significance_level.
    
    """
    X = sm.add_constant(X)  # add intercept
    model = sm.OLS(y, X).fit()
    
    while True:
        # Get max p-value
        p_values = model.pvalues
        max_pval = p_values.max()
        worst_var = p_values.idxmax()
        
        # Stop if all p-values are <= significance_level
        if max_pval <= significance_level:
            break
        
        # Do not drop the constant
        if worst_var == "const":
            break
        
        # Drop worst variable
        print(f"Dropping '{worst_var}' (p-value = {max_pval:.4f})")
        X = X.drop(columns=[worst_var])
        
        # Refit model
        model = sm.OLS(y, X).fit()
    
    return model, X

# Benin

In [33]:
# Upload the excel file and save 
x ="/Users/harold/DataAnalyctisandScience/Cointegration_Johansen_test/dataTransportCPI/Benin_completes"
df = pd.read_excel(x+".xlsx")

In [34]:
# Define X and y
X = df.drop(columns=["transportation CPI", "Months"])   
X = sm.add_constant(X)       
y = df["transportation CPI"]
ols_model, X_selected = backward_elimination(y, X, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'Crude oil, WTI ($/bbl) CRUDE_WTI' (p-value = 0.7264)
Dropping 'Crude oil, Dubai ($/bbl) CRUDE_DUBAI' (p-value = 0.9611)
Dropping 'Crude oil, average ($/bbl) CRUDE_PETRO' (p-value = 0.9520)
Dropping 'Crude oil, Brent ($/bbl) CRUDE_BRENT' (p-value = 0.7734)
Dropping 'Natural gas, Europe ($/mmbtu) NGAS_EUR' (p-value = 0.4551)
Dropping 'Natural gas, US ($/mmbtu) NGAS_US' (p-value = 0.1839)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.112
Model:                            OLS   Adj. R-squared:                  0.103
Method:                 Least Squares   F-statistic:                     12.23
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           1.47e-07
Time:                        13:03:05   Log-Likelihood:                 285.59
No. Observations:                 294   AIC:                            -563.2
Df Residuals:             

In [35]:
# 1. Create lag of y
y_lagged = y.shift(1).rename("lag_y")

# 2. Combine everything
data = pd.concat([y, y_lagged, X], axis=1).dropna()


# 3. Current y
y_current = data[y.name]

# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X.columns]], axis=1))

# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())

                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.706
Model:                            OLS   Adj. R-squared:                  0.696
Method:                 Least Squares   F-statistic:                     67.79
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           4.34e-69
Time:                        13:03:05   Log-Likelihood:                 452.89
No. Observations:                 293   AIC:                            -883.8
Df Residuals:                     282   BIC:                            -843.3
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                                                     coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------

In [36]:
ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'Natural gas, US ($/mmbtu) NGAS_US' (p-value = 0.8593)
Dropping 'Natural gas, Europe ($/mmbtu) NGAS_EUR' (p-value = 0.8849)
Dropping 'Crude oil, Dubai ($/bbl) CRUDE_DUBAI' (p-value = 0.5738)
Dropping 'Crude oil, Brent ($/bbl) CRUDE_BRENT' (p-value = 0.3576)
Dropping 'Crude oil, WTI ($/bbl) CRUDE_WTI' (p-value = 0.5605)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.705
Model:                            OLS   Adj. R-squared:                  0.699
Method:                 Least Squares   F-statistic:                     136.9
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           7.77e-74
Time:                        13:03:05   Log-Likelihood:                 452.09
No. Observations:                 293   AIC:                            -892.2
Df Residuals:                     287   BIC:                            -870.1
Df Model:       

In [37]:
# 1. Create lag of X

X_lagged = X.shift(1).drop(columns=['const']).rename(columns=lambda x: f"lag_{x}")
const = X['const'].shift(1)
X_lagged = pd.concat([const, X_lagged], axis=1).dropna()

model_2 = sm.OLS(y_current, X_lagged).fit()
print(model_2.summary())

                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.154
Model:                            OLS   Adj. R-squared:                  0.127
Method:                 Least Squares   F-statistic:                     5.720
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           2.70e-07
Time:                        13:03:05   Log-Likelihood:                 297.92
No. Observations:                 293   AIC:                            -575.8
Df Residuals:                     283   BIC:                            -539.0
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                                                         coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------

In [38]:
ols_model, X_selected = backward_elimination(y_current, X_lagged, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'lag_Natural gas, Europe ($/mmbtu) NGAS_EUR' (p-value = 0.5903)
Dropping 'lag_Crude oil, Dubai ($/bbl) CRUDE_DUBAI' (p-value = 0.4038)
Dropping 'lag_Crude oil, Brent ($/bbl) CRUDE_BRENT' (p-value = 0.4770)
Dropping 'lag_Crude oil, WTI ($/bbl) CRUDE_WTI' (p-value = 0.5003)
Dropping 'lag_Natural gas, US ($/mmbtu) NGAS_US' (p-value = 0.3136)
Dropping 'lag_Crude oil, average ($/bbl) CRUDE_PETRO' (p-value = 0.1074)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.137
Model:                            OLS   Adj. R-squared:                  0.128
Method:                 Least Squares   F-statistic:                     15.33
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           2.77e-09
Time:                        13:03:05   Log-Likelihood:                 295.07
No. Observations:                 293   AIC:                            -582.1
Df

In [39]:
# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")
X_lagged = X.shift(1).drop(columns=['const']).rename(columns=lambda x: f"lag_{x}")



# 2. Combine everything
data = pd.concat([y, y_lagged, X_lagged], axis=1).dropna()


# 3. Current y
y_current = data[y.name]   # dependent variable


# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X_lagged.columns]], axis=1))


# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())

                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.710
Model:                            OLS   Adj. R-squared:                  0.699
Method:                 Least Squares   F-statistic:                     68.92
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           8.57e-70
Time:                        13:03:06   Log-Likelihood:                 454.60
No. Observations:                 293   AIC:                            -887.2
Df Residuals:                     282   BIC:                            -846.7
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                                                         coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------

In [40]:
ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'lag_Natural gas, Europe ($/mmbtu) NGAS_EUR' (p-value = 0.9743)
Dropping 'lag_Natural gas, US ($/mmbtu) NGAS_US' (p-value = 0.7645)
Dropping 'lag_Coal, South African **($/mt) COAL_SAFRICA' (p-value = 0.4648)
Dropping 'lag_Crude oil, Dubai ($/bbl) CRUDE_DUBAI' (p-value = 0.4217)
Dropping 'lag_Crude oil, Brent ($/bbl) CRUDE_BRENT' (p-value = 0.2412)
Dropping 'lag_Crude oil, WTI ($/bbl) CRUDE_WTI' (p-value = 0.2550)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.706
Model:                            OLS   Adj. R-squared:                  0.702
Method:                 Least Squares   F-statistic:                     172.6
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           3.47e-75
Time:                        13:03:06   Log-Likelihood:                 452.57
No. Observations:                 293   AIC:                            -895.1

# Burkina

In [41]:
# Upload the excel file and save 
x ="/Users/harold/DataAnalyctisandScience/Cointegration_Johansen_test/dataTransportCPI/Burkina_completes"
df = pd.read_excel(x+".xlsx")


In [42]:

# Define X and y
X = df.drop(columns=["transportation CPI", "Months"])   
X = sm.add_constant(X)       
y = df["transportation CPI"]
ols_model, X_selected = backward_elimination(y, X, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())



Dropping 'Coal, South African **($/mt) COAL_SAFRICA' (p-value = 0.9502)
Dropping 'Crude oil, WTI ($/bbl) CRUDE_WTI' (p-value = 0.8637)
Dropping 'Crude oil, Dubai ($/bbl) CRUDE_DUBAI' (p-value = 0.9086)
Dropping 'Crude oil, average ($/bbl) CRUDE_PETRO' (p-value = 0.4294)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.278
Model:                            OLS   Adj. R-squared:                  0.265
Method:                 Least Squares   F-statistic:                     22.13
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           9.22e-19
Time:                        13:03:06   Log-Likelihood:                 533.11
No. Observations:                 294   AIC:                            -1054.
Df Residuals:                     288   BIC:                            -1032.
Df Model:                           5                                      

In [43]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")

# 2. Combine everything
data = pd.concat([y, y_lagged, X], axis=1).dropna()


# 3. Current y
y_current = data[y.name]

# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X.columns]], axis=1))

# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.838
Model:                            OLS   Adj. R-squared:                  0.832
Method:                 Least Squares   F-statistic:                     145.8
Date:                Tue, 21 Oct 2025   Prob (F-statistic):          3.22e-105
Time:                        13:03:06   Log-Likelihood:                 753.17
No. Observations:                 293   AIC:                            -1484.
Df Residuals:                     282   BIC:                            -1444.
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                                                     coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------

In [44]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'Coal, South African **($/mt) COAL_SAFRICA' (p-value = 0.8380)
Dropping 'Crude oil, Brent ($/bbl) CRUDE_BRENT' (p-value = 0.8265)
Dropping 'Coal, Australian ($/mt) COAL_AUS' (p-value = 0.2895)
Dropping 'Liquefied natural gas, Japan ($/mmbtu) NGAS_JP' (p-value = 0.2985)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.837
Model:                            OLS   Adj. R-squared:                  0.833
Method:                 Least Squares   F-statistic:                     244.1
Date:                Tue, 21 Oct 2025   Prob (F-statistic):          2.24e-109
Time:                        13:03:06   Log-Likelihood:                 751.99
No. Observations:                 293   AIC:                            -1490.
Df Residuals:                     286   BIC:                            -1464.
Df Model:                           6                              

In [45]:

# 1. Create lag of X

X_lagged = X.shift(1).drop(columns=['const']).rename(columns=lambda x: f"lag_{x}")
const = X['const'].shift(1)
X_lagged = pd.concat([const, X_lagged], axis=1).dropna()

model_2 = sm.OLS(y_current, X_lagged).fit()
print(model_2.summary())



                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.329
Model:                            OLS   Adj. R-squared:                  0.308
Method:                 Least Squares   F-statistic:                     15.42
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           1.97e-20
Time:                        13:03:06   Log-Likelihood:                 545.04
No. Observations:                 293   AIC:                            -1070.
Df Residuals:                     283   BIC:                            -1033.
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                                                         coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------

In [46]:

ols_model, X_selected = backward_elimination(y_current, X_lagged, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())



Dropping 'lag_Crude oil, WTI ($/bbl) CRUDE_WTI' (p-value = 0.9909)
Dropping 'lag_Crude oil, Dubai ($/bbl) CRUDE_DUBAI' (p-value = 0.7383)
Dropping 'lag_Coal, South African **($/mt) COAL_SAFRICA' (p-value = 0.5596)
Dropping 'lag_Crude oil, average ($/bbl) CRUDE_PETRO' (p-value = 0.1695)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.324
Model:                            OLS   Adj. R-squared:                  0.312
Method:                 Least Squares   F-statistic:                     27.45
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           1.07e-22
Time:                        13:03:06   Log-Likelihood:                 543.84
No. Observations:                 293   AIC:                            -1076.
Df Residuals:                     287   BIC:                            -1054.
Df Model:                           5                      

In [47]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")
X_lagged = X.shift(1).drop(columns=['const']).rename(columns=lambda x: f"lag_{x}")


# 2. Combine everything
data = pd.concat([y, y_lagged, X_lagged], axis=1).dropna()


# 3. Current y
y_current = data[y.name]   # dependent variable


# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X_lagged.columns]], axis=1))


# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.839
Model:                            OLS   Adj. R-squared:                  0.834
Method:                 Least Squares   F-statistic:                     147.2
Date:                Tue, 21 Oct 2025   Prob (F-statistic):          1.03e-105
Time:                        13:03:06   Log-Likelihood:                 754.36
No. Observations:                 293   AIC:                            -1487.
Df Residuals:                     282   BIC:                            -1446.
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                                                         coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------

In [48]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'lag_Crude oil, average ($/bbl) CRUDE_PETRO' (p-value = 0.8916)
Dropping 'lag_Crude oil, Dubai ($/bbl) CRUDE_DUBAI' (p-value = 0.7193)
Dropping 'lag_Liquefied natural gas, Japan ($/mmbtu) NGAS_JP' (p-value = 0.5108)
Dropping 'lag_Natural gas, Europe ($/mmbtu) NGAS_EUR' (p-value = 0.2722)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.838
Model:                            OLS   Adj. R-squared:                  0.835
Method:                 Least Squares   F-statistic:                     247.0
Date:                Tue, 21 Oct 2025   Prob (F-statistic):          5.46e-110
Time:                        13:03:06   Log-Likelihood:                 753.44
No. Observations:                 293   AIC:                            -1493.
Df Residuals:                     286   BIC:                            -1467.
Df Model:                           6           

# Côte d'Ivoire

In [49]:
# Upload the excel file and save 
x ="/Users/harold/DataAnalyctisandScience/Cointegration_Johansen_test/dataTransportCPI/Ivoire"
df = pd.read_excel(x+".xlsx")


In [50]:

# Define X and y
X = df.drop(columns=["transportation CPI", "Months"])   
X = sm.add_constant(X)       
y = df["transportation CPI"]
ols_model, X_selected = backward_elimination(y, X, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())



Dropping 'Coal, Australian ($/mt) COAL_AUS' (p-value = 0.4561)
Dropping 'Crude oil, Dubai ($/bbl) CRUDE_DUBAI' (p-value = 0.2052)
Dropping 'Crude oil, WTI ($/bbl) CRUDE_WTI' (p-value = 0.1056)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.691
Model:                            OLS   Adj. R-squared:                  0.684
Method:                 Least Squares   F-statistic:                     111.2
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           3.84e-73
Time:                        13:03:06   Log-Likelihood:                -1041.7
No. Observations:                 306   AIC:                             2097.
Df Residuals:                     299   BIC:                             2123.
Df Model:                           6                                         
Covariance Type:            nonrobust                                     

In [51]:

# 1. Create lag of y
y_lagged = y.shift(1).rename("lag_y")

# 2. Combine everything
data = pd.concat([y, y_lagged, X], axis=1).dropna()


# 3. Current y
y_current = data[y.name]

# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X.columns]], axis=1))

# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.987
Model:                            OLS   Adj. R-squared:                  0.987
Method:                 Least Squares   F-statistic:                     2314.
Date:                Tue, 21 Oct 2025   Prob (F-statistic):          5.90e-273
Time:                        13:03:06   Log-Likelihood:                -547.41
No. Observations:                 305   AIC:                             1117.
Df Residuals:                     294   BIC:                             1158.
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                                                     coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------

In [52]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'Natural gas, US ($/mmbtu) NGAS_US' (p-value = 0.8600)
Dropping 'Crude oil, Dubai ($/bbl) CRUDE_DUBAI' (p-value = 0.4940)
Dropping 'Crude oil, WTI ($/bbl) CRUDE_WTI' (p-value = 0.9621)
Dropping 'Crude oil, Brent ($/bbl) CRUDE_BRENT' (p-value = 0.7007)
Dropping 'Liquefied natural gas, Japan ($/mmbtu) NGAS_JP' (p-value = 0.1502)
Dropping 'Crude oil, average ($/bbl) CRUDE_PETRO' (p-value = 0.4422)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.987
Model:                            OLS   Adj. R-squared:                  0.987
Method:                 Least Squares   F-statistic:                     5838.
Date:                Tue, 21 Oct 2025   Prob (F-statistic):          4.59e-283
Time:                        13:03:06   Log-Likelihood:                -549.09
No. Observations:                 305   AIC:                             1108.
Df Residuals:     

In [53]:

# 1. Create lag of X

X_lagged = X.shift(1).drop(columns=['const']).rename(columns=lambda x: f"lag_{x}")
const = X['const'].shift(1)
X_lagged = pd.concat([const, X_lagged], axis=1).dropna()

model_2 = sm.OLS(y_current, X_lagged).fit()
print(model_2.summary())



                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.693
Model:                            OLS   Adj. R-squared:                  0.684
Method:                 Least Squares   F-statistic:                     74.03
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           2.12e-70
Time:                        13:03:06   Log-Likelihood:                -1035.0
No. Observations:                 305   AIC:                             2090.
Df Residuals:                     295   BIC:                             2127.
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                                                         coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------

In [54]:

ols_model, X_selected = backward_elimination(y_current, X_lagged, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())



Dropping 'lag_Coal, Australian ($/mt) COAL_AUS' (p-value = 0.2735)
Dropping 'lag_Crude oil, Dubai ($/bbl) CRUDE_DUBAI' (p-value = 0.2560)
Dropping 'lag_Crude oil, WTI ($/bbl) CRUDE_WTI' (p-value = 0.1118)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.688
Model:                            OLS   Adj. R-squared:                  0.682
Method:                 Least Squares   F-statistic:                     109.4
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           2.48e-72
Time:                        13:03:06   Log-Likelihood:                -1037.6
No. Observations:                 305   AIC:                             2089.
Df Residuals:                     298   BIC:                             2115.
Df Model:                           6                                         
Covariance Type:            nonrobust                         

In [55]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")
X_lagged = X.shift(1).drop(columns=['const']).rename(columns=lambda x: f"lag_{x}")


# 2. Combine everything
data = pd.concat([y, y_lagged, X_lagged], axis=1).dropna()


# 3. Current y
y_current = data[y.name]   # dependent variable


# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X_lagged.columns]], axis=1))


# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.987
Model:                            OLS   Adj. R-squared:                  0.987
Method:                 Least Squares   F-statistic:                     2271.
Date:                Tue, 21 Oct 2025   Prob (F-statistic):          9.17e-272
Time:                        13:03:07   Log-Likelihood:                -550.25
No. Observations:                 305   AIC:                             1123.
Df Residuals:                     294   BIC:                             1163.
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                                                         coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------

In [56]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'lag_Crude oil, Brent ($/bbl) CRUDE_BRENT' (p-value = 0.9040)
Dropping 'lag_Crude oil, Dubai ($/bbl) CRUDE_DUBAI' (p-value = 0.9924)
Dropping 'lag_Crude oil, average ($/bbl) CRUDE_PETRO' (p-value = 0.8560)
Dropping 'lag_Natural gas, Europe ($/mmbtu) NGAS_EUR' (p-value = 0.6097)
Dropping 'lag_Natural gas, US ($/mmbtu) NGAS_US' (p-value = 0.6262)
Dropping 'lag_Liquefied natural gas, Japan ($/mmbtu) NGAS_JP' (p-value = 0.2611)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.987
Model:                            OLS   Adj. R-squared:                  0.987
Method:                 Least Squares   F-statistic:                     5757.
Date:                Tue, 21 Oct 2025   Prob (F-statistic):          3.58e-282
Time:                        13:03:07   Log-Likelihood:                -551.18
No. Observations:                 305   AIC:                       

# Guinea

In [57]:
# Upload the excel file and save 
x ="/Users/harold/DataAnalyctisandScience/Cointegration_Johansen_test/dataTransportCPI/Guinea_completes"
df = pd.read_excel(x+".xlsx")


In [58]:

# Define X and y
X = df.drop(columns=["transportation CPI", "Months"])   
X = sm.add_constant(X)       
y = df["transportation CPI"]
ols_model, X_selected = backward_elimination(y, X, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())



Dropping 'Natural gas, Europe ($/mmbtu) NGAS_EUR' (p-value = 0.3775)
Dropping 'Liquefied natural gas, Japan ($/mmbtu) NGAS_JP' (p-value = 0.1721)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.237
Model:                            OLS   Adj. R-squared:                  0.216
Method:                 Least Squares   F-statistic:                     11.33
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           1.66e-12
Time:                        13:03:07   Log-Likelihood:                 417.07
No. Observations:                 264   AIC:                            -818.1
Df Residuals:                     256   BIC:                            -789.5
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                                          

In [59]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")

# 2. Combine everything
data = pd.concat([y, y_lagged, X], axis=1).dropna()


# 3. Current y
y_current = data[y.name]

# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X.columns]], axis=1))

# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.709
Model:                            OLS   Adj. R-squared:                  0.698
Method:                 Least Squares   F-statistic:                     61.45
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           7.65e-62
Time:                        13:03:07   Log-Likelihood:                 541.92
No. Observations:                 263   AIC:                            -1062.
Df Residuals:                     252   BIC:                            -1023.
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                                                     coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------

In [60]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'Natural gas, Europe ($/mmbtu) NGAS_EUR' (p-value = 0.8873)
Dropping 'Liquefied natural gas, Japan ($/mmbtu) NGAS_JP' (p-value = 0.8499)
Dropping 'Coal, South African **($/mt) COAL_SAFRICA' (p-value = 0.7495)
Dropping 'Coal, Australian ($/mt) COAL_AUS' (p-value = 0.9121)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.709
Model:                            OLS   Adj. R-squared:                  0.702
Method:                 Least Squares   F-statistic:                     103.9
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           1.03e-65
Time:                        13:03:07   Log-Likelihood:                 541.83
No. Observations:                 263   AIC:                            -1070.
Df Residuals:                     256   BIC:                            -1045.
Df Model:                           6                            

In [61]:

# 1. Create lag of X

X_lagged = X.shift(1).drop(columns=['const']).rename(columns=lambda x: f"lag_{x}")
const = X['const'].shift(1)
X_lagged = pd.concat([const, X_lagged], axis=1).dropna()

model_2 = sm.OLS(y_current, X_lagged).fit()
print(model_2.summary())



                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.212
Model:                            OLS   Adj. R-squared:                  0.184
Method:                 Least Squares   F-statistic:                     7.543
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           8.79e-10
Time:                        13:03:07   Log-Likelihood:                 410.78
No. Observations:                 263   AIC:                            -801.6
Df Residuals:                     253   BIC:                            -765.8
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                                                         coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------

In [62]:

ols_model, X_selected = backward_elimination(y_current, X_lagged, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())



Dropping 'lag_Natural gas, Europe ($/mmbtu) NGAS_EUR' (p-value = 0.5545)
Dropping 'lag_Liquefied natural gas, Japan ($/mmbtu) NGAS_JP' (p-value = 0.1033)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.202
Model:                            OLS   Adj. R-squared:                  0.180
Method:                 Least Squares   F-statistic:                     9.230
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           3.54e-10
Time:                        13:03:07   Log-Likelihood:                 409.22
No. Observations:                 263   AIC:                            -802.4
Df Residuals:                     255   BIC:                            -773.9
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                                  

In [63]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")
X_lagged = X.shift(1).drop(columns=['const']).rename(columns=lambda x: f"lag_{x}")


# 2. Combine everything
data = pd.concat([y, y_lagged, X_lagged], axis=1).dropna()


# 3. Current y
y_current = data[y.name]   # dependent variable


# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X_lagged.columns]], axis=1))


# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.715
Model:                            OLS   Adj. R-squared:                  0.704
Method:                 Least Squares   F-statistic:                     63.27
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           5.75e-63
Time:                        13:03:07   Log-Likelihood:                 544.66
No. Observations:                 263   AIC:                            -1067.
Df Residuals:                     252   BIC:                            -1028.
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                                                         coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------

In [64]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'lag_Crude oil, Brent ($/bbl) CRUDE_BRENT' (p-value = 0.9434)
Dropping 'lag_Natural gas, Europe ($/mmbtu) NGAS_EUR' (p-value = 0.8552)
Dropping 'lag_Coal, South African **($/mt) COAL_SAFRICA' (p-value = 0.7345)
Dropping 'lag_Liquefied natural gas, Japan ($/mmbtu) NGAS_JP' (p-value = 0.3651)
Dropping 'lag_Coal, Australian ($/mt) COAL_AUS' (p-value = 0.2931)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.713
Model:                            OLS   Adj. R-squared:                  0.707
Method:                 Least Squares   F-statistic:                     127.6
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           1.59e-67
Time:                        13:03:07   Log-Likelihood:                 543.59
No. Observations:                 263   AIC:                            -1075.
Df Residuals:                     257   BIC:             

# Mali 

In [65]:
# Upload the excel file and save 
x ="/Users/harold/DataAnalyctisandScience/Cointegration_Johansen_test/dataTransportCPI/Mali_completes"
df = pd.read_excel(x+".xlsx")


In [66]:

# Define X and y
X = df.drop(columns=["transportation CPI", "Months"])   
X = sm.add_constant(X)       
y = df["transportation CPI"]
ols_model, X_selected = backward_elimination(y, X, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())



Dropping 'Coal, South African **($/mt) COAL_SAFRICA' (p-value = 0.7330)
Dropping 'Crude oil, WTI ($/bbl) CRUDE_WTI' (p-value = 0.2117)
Dropping 'Crude oil, Brent ($/bbl) CRUDE_BRENT' (p-value = 0.7745)
Dropping 'Crude oil, average ($/bbl) CRUDE_PETRO' (p-value = 0.1263)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.420
Model:                            OLS   Adj. R-squared:                  0.410
Method:                 Least Squares   F-statistic:                     41.78
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           2.83e-32
Time:                        13:03:07   Log-Likelihood:                 596.56
No. Observations:                 294   AIC:                            -1181.
Df Residuals:                     288   BIC:                            -1159.
Df Model:                           5                                      

In [67]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")

# 2. Combine everything
data = pd.concat([y, y_lagged, X], axis=1).dropna()


# 3. Current y
y_current = data[y.name]

# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X.columns]], axis=1))

# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.700
Model:                            OLS   Adj. R-squared:                  0.689
Method:                 Least Squares   F-statistic:                     65.81
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           7.98e-68
Time:                        13:03:07   Log-Likelihood:                 695.62
No. Observations:                 293   AIC:                            -1369.
Df Residuals:                     282   BIC:                            -1329.
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                                                     coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------

In [68]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'Crude oil, Brent ($/bbl) CRUDE_BRENT' (p-value = 0.6042)
Dropping 'Crude oil, WTI ($/bbl) CRUDE_WTI' (p-value = 0.5474)
Dropping 'Natural gas, Europe ($/mmbtu) NGAS_EUR' (p-value = 0.4347)
Dropping 'Coal, South African **($/mt) COAL_SAFRICA' (p-value = 0.2756)
Dropping 'Crude oil, average ($/bbl) CRUDE_PETRO' (p-value = 0.2349)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.696
Model:                            OLS   Adj. R-squared:                  0.691
Method:                 Least Squares   F-statistic:                     131.4
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           4.86e-72
Time:                        13:03:07   Log-Likelihood:                 693.64
No. Observations:                 293   AIC:                            -1375.
Df Residuals:                     287   BIC:                            -1353.
Df Mod

In [69]:

# 1. Create lag of X

X_lagged = X.shift(1).drop(columns=['const']).rename(columns=lambda x: f"lag_{x}")
const = X['const'].shift(1)
X_lagged = pd.concat([const, X_lagged], axis=1).dropna()

model_2 = sm.OLS(y_current, X_lagged).fit()
print(model_2.summary())



                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.456
Model:                            OLS   Adj. R-squared:                  0.439
Method:                 Least Squares   F-statistic:                     26.38
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           7.40e-33
Time:                        13:03:07   Log-Likelihood:                 608.47
No. Observations:                 293   AIC:                            -1197.
Df Residuals:                     283   BIC:                            -1160.
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                                                         coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------

In [70]:

ols_model, X_selected = backward_elimination(y_current, X_lagged, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())



Dropping 'lag_Coal, South African **($/mt) COAL_SAFRICA' (p-value = 0.4897)
Dropping 'lag_Crude oil, WTI ($/bbl) CRUDE_WTI' (p-value = 0.3638)
Dropping 'lag_Crude oil, Brent ($/bbl) CRUDE_BRENT' (p-value = 0.3485)
Dropping 'lag_Crude oil, average ($/bbl) CRUDE_PETRO' (p-value = 0.1690)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.448
Model:                            OLS   Adj. R-squared:                  0.439
Method:                 Least Squares   F-statistic:                     46.66
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           3.36e-35
Time:                        13:03:07   Log-Likelihood:                 606.38
No. Observations:                 293   AIC:                            -1201.
Df Residuals:                     287   BIC:                            -1179.
Df Model:                           5                      

In [71]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")
X_lagged = X.shift(1).drop(columns=['const']).rename(columns=lambda x: f"lag_{x}")


# 2. Combine everything
data = pd.concat([y, y_lagged, X_lagged], axis=1).dropna()


# 3. Current y
y_current = data[y.name]   # dependent variable


# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X_lagged.columns]], axis=1))


# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.695
Model:                            OLS   Adj. R-squared:                  0.685
Method:                 Least Squares   F-statistic:                     64.37
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           6.78e-67
Time:                        13:03:08   Log-Likelihood:                 693.37
No. Observations:                 293   AIC:                            -1365.
Df Residuals:                     282   BIC:                            -1324.
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                                                         coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------

In [72]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'lag_Crude oil, WTI ($/bbl) CRUDE_WTI' (p-value = 0.9289)
Dropping 'lag_Coal, South African **($/mt) COAL_SAFRICA' (p-value = 0.5632)
Dropping 'lag_Coal, Australian ($/mt) COAL_AUS' (p-value = 0.3624)
Dropping 'lag_Crude oil, Brent ($/bbl) CRUDE_BRENT' (p-value = 0.3859)
Dropping 'lag_Crude oil, average ($/bbl) CRUDE_PETRO' (p-value = 0.4933)
Dropping 'lag_Natural gas, Europe ($/mmbtu) NGAS_EUR' (p-value = 0.1438)
Dropping 'lag_Liquefied natural gas, Japan ($/mmbtu) NGAS_JP' (p-value = 0.2170)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.689
Model:                            OLS   Adj. R-squared:                  0.686
Method:                 Least Squares   F-statistic:                     213.3
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           6.10e-73
Time:                        13:03:08   Log-Likelihood:                 690

# Niger

In [73]:
# Upload the excel file and save 
x ="/Users/harold/DataAnalyctisandScience/Cointegration_Johansen_test/dataTransportCPI/Niger_completes"
df = pd.read_excel(x+".xlsx")


In [74]:

# Define X and y
X = df.drop(columns=["transportation CPI", "Months"])   
X = sm.add_constant(X)       
y = df["transportation CPI"]
ols_model, X_selected = backward_elimination(y, X, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())




Final model summary:
                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.338
Model:                            OLS   Adj. R-squared:                  0.317
Method:                 Least Squares   F-statistic:                     16.11
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           2.65e-21
Time:                        13:03:08   Log-Likelihood:                 488.40
No. Observations:                 294   AIC:                            -956.8
Df Residuals:                     284   BIC:                            -920.0
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                                                     coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------

In [75]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")

# 2. Combine everything
data = pd.concat([y, y_lagged, X], axis=1).dropna()


# 3. Current y
y_current = data[y.name]

# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X.columns]], axis=1))

# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.826
Model:                            OLS   Adj. R-squared:                  0.820
Method:                 Least Squares   F-statistic:                     134.1
Date:                Tue, 21 Oct 2025   Prob (F-statistic):          5.81e-101
Time:                        13:03:08   Log-Likelihood:                 685.83
No. Observations:                 293   AIC:                            -1350.
Df Residuals:                     282   BIC:                            -1309.
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                                                     coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------

In [76]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'Liquefied natural gas, Japan ($/mmbtu) NGAS_JP' (p-value = 0.9984)
Dropping 'Crude oil, Brent ($/bbl) CRUDE_BRENT' (p-value = 0.7931)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.826
Model:                            OLS   Adj. R-squared:                  0.821
Method:                 Least Squares   F-statistic:                     168.7
Date:                Tue, 21 Oct 2025   Prob (F-statistic):          3.56e-103
Time:                        13:03:08   Log-Likelihood:                 685.80
No. Observations:                 293   AIC:                            -1354.
Df Residuals:                     284   BIC:                            -1320.
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                                            

In [77]:
# 1. Create lag of X

X_lagged = X.shift(1).drop(columns=['const']).rename(columns=lambda x: f"lag_{x}")
const = X['const'].shift(1)
X_lagged = pd.concat([const, X_lagged], axis=1).dropna()

model_2 = sm.OLS(y_current, X_lagged).fit()
print(model_2.summary())



                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.374
Model:                            OLS   Adj. R-squared:                  0.354
Method:                 Least Squares   F-statistic:                     18.81
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           1.58e-24
Time:                        13:03:08   Log-Likelihood:                 498.16
No. Observations:                 293   AIC:                            -976.3
Df Residuals:                     283   BIC:                            -939.5
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                                                         coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------

In [78]:

ols_model, X_selected = backward_elimination(y_current, X_lagged, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())




Final model summary:
                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.374
Model:                            OLS   Adj. R-squared:                  0.354
Method:                 Least Squares   F-statistic:                     18.81
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           1.58e-24
Time:                        13:03:08   Log-Likelihood:                 498.16
No. Observations:                 293   AIC:                            -976.3
Df Residuals:                     283   BIC:                            -939.5
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                                                         coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------

In [79]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")
X_lagged = X.shift(1).drop(columns=['const']).rename(columns=lambda x: f"lag_{x}")


# 2. Combine everything
data = pd.concat([y, y_lagged, X_lagged], axis=1).dropna()


# 3. Current y
y_current = data[y.name]   # dependent variable


# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X_lagged.columns]], axis=1))


# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.822
Model:                            OLS   Adj. R-squared:                  0.816
Method:                 Least Squares   F-statistic:                     130.3
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           1.53e-99
Time:                        13:03:08   Log-Likelihood:                 682.41
No. Observations:                 293   AIC:                            -1343.
Df Residuals:                     282   BIC:                            -1302.
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                                                         coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------

In [80]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Final model summary:
                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.822
Model:                            OLS   Adj. R-squared:                  0.816
Method:                 Least Squares   F-statistic:                     130.3
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           1.53e-99
Time:                        13:03:08   Log-Likelihood:                 682.41
No. Observations:                 293   AIC:                            -1343.
Df Residuals:                     282   BIC:                            -1302.
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                                                         coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------

# Senegal

In [81]:
# Upload the excel file and save 
x ="/Users/harold/DataAnalyctisandScience/Cointegration_Johansen_test/dataTransportCPI/Senegal_completes"
df = pd.read_excel(x+".xlsx")


In [82]:

# Define X and y
X = df.drop(columns=["transportation CPI", "Months"])   
X = sm.add_constant(X)       
y = df["transportation CPI"]
ols_model, X_selected = backward_elimination(y, X, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())




Final model summary:
                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.425
Model:                            OLS   Adj. R-squared:                  0.407
Method:                 Least Squares   F-statistic:                     23.31
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           1.22e-29
Time:                        13:03:08   Log-Likelihood:                 600.35
No. Observations:                 294   AIC:                            -1181.
Df Residuals:                     284   BIC:                            -1144.
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                                                     coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------

In [83]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")

# 2. Combine everything
data = pd.concat([y, y_lagged, X], axis=1).dropna()


# 3. Current y
y_current = data[y.name]

# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X.columns]], axis=1))

# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.907
Model:                            OLS   Adj. R-squared:                  0.903
Method:                 Least Squares   F-statistic:                     273.7
Date:                Tue, 21 Oct 2025   Prob (F-statistic):          7.81e-139
Time:                        13:03:08   Log-Likelihood:                 864.96
No. Observations:                 293   AIC:                            -1708.
Df Residuals:                     282   BIC:                            -1667.
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                                                     coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------

In [84]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())



Final model summary:
                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.907
Model:                            OLS   Adj. R-squared:                  0.903
Method:                 Least Squares   F-statistic:                     273.7
Date:                Tue, 21 Oct 2025   Prob (F-statistic):          7.81e-139
Time:                        13:03:08   Log-Likelihood:                 864.96
No. Observations:                 293   AIC:                            -1708.
Df Residuals:                     282   BIC:                            -1667.
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                                                     coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------

In [85]:
# 1. Create lag of X

X_lagged = X.shift(1).drop(columns=['const']).rename(columns=lambda x: f"lag_{x}")
const = X['const'].shift(1)
X_lagged = pd.concat([const, X_lagged], axis=1).dropna()

model_2 = sm.OLS(y_current, X_lagged).fit()
print(model_2.summary())



                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.436
Model:                            OLS   Adj. R-squared:                  0.418
Method:                 Least Squares   F-statistic:                     24.31
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           1.10e-30
Time:                        13:03:09   Log-Likelihood:                 601.52
No. Observations:                 293   AIC:                            -1183.
Df Residuals:                     283   BIC:                            -1146.
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                                                         coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------

In [86]:

ols_model, X_selected = backward_elimination(y_current, X_lagged, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())



Dropping 'lag_Crude oil, Brent ($/bbl) CRUDE_BRENT' (p-value = 0.2748)
Dropping 'lag_Crude oil, WTI ($/bbl) CRUDE_WTI' (p-value = 0.5846)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.433
Model:                            OLS   Adj. R-squared:                  0.419
Method:                 Least Squares   F-statistic:                     31.10
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           7.24e-32
Time:                        13:03:09   Log-Likelihood:                 600.75
No. Observations:                 293   AIC:                            -1186.
Df Residuals:                     285   BIC:                            -1156.
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                                                  

In [87]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")
X_lagged = X.shift(1).drop(columns=['const']).rename(columns=lambda x: f"lag_{x}")


# 2. Combine everything
data = pd.concat([y, y_lagged, X_lagged], axis=1).dropna()


# 3. Current y
y_current = data[y.name]   # dependent variable


# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X_lagged.columns]], axis=1))


# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.904
Model:                            OLS   Adj. R-squared:                  0.901
Method:                 Least Squares   F-statistic:                     265.5
Date:                Tue, 21 Oct 2025   Prob (F-statistic):          3.79e-137
Time:                        13:03:09   Log-Likelihood:                 860.92
No. Observations:                 293   AIC:                            -1700.
Df Residuals:                     282   BIC:                            -1659.
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                                                         coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------

In [88]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Final model summary:
                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.904
Model:                            OLS   Adj. R-squared:                  0.901
Method:                 Least Squares   F-statistic:                     265.5
Date:                Tue, 21 Oct 2025   Prob (F-statistic):          3.79e-137
Time:                        13:03:09   Log-Likelihood:                 860.92
No. Observations:                 293   AIC:                            -1700.
Df Residuals:                     282   BIC:                            -1659.
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                                                         coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------

# Togo

In [89]:
# Upload the excel file and save 
x ="/Users/harold/DataAnalyctisandScience/Cointegration_Johansen_test/dataTransportCPI/Togo_completes"
df = pd.read_excel(x+".xlsx")


In [90]:

# Define X and y
X = df.drop(columns=["transportation CPI", "Months"])   
X = sm.add_constant(X)       
y = df["transportation CPI"]
ols_model, X_selected = backward_elimination(y, X, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())



Dropping 'Coal, Australian ($/mt) COAL_AUS' (p-value = 0.5710)
Dropping 'Natural gas, Europe ($/mmbtu) NGAS_EUR' (p-value = 0.4633)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.384
Model:                            OLS   Adj. R-squared:                  0.369
Method:                 Least Squares   F-statistic:                     25.52
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           5.24e-27
Time:                        13:03:09   Log-Likelihood:                 421.15
No. Observations:                 294   AIC:                            -826.3
Df Residuals:                     286   BIC:                            -796.8
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                                                     coe

In [91]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")

# 2. Combine everything
data = pd.concat([y, y_lagged, X], axis=1).dropna()


# 3. Current y
y_current = data[y.name]

# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X.columns]], axis=1))

# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.874
Model:                            OLS   Adj. R-squared:                  0.870
Method:                 Least Squares   F-statistic:                     195.6
Date:                Tue, 21 Oct 2025   Prob (F-statistic):          1.45e-120
Time:                        13:03:09   Log-Likelihood:                 652.00
No. Observations:                 293   AIC:                            -1282.
Df Residuals:                     282   BIC:                            -1242.
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                                                     coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------

In [92]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'Natural gas, Europe ($/mmbtu) NGAS_EUR' (p-value = 0.8359)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.874
Model:                            OLS   Adj. R-squared:                  0.870
Method:                 Least Squares   F-statistic:                     218.1
Date:                Tue, 21 Oct 2025   Prob (F-statistic):          9.76e-122
Time:                        13:03:09   Log-Likelihood:                 651.97
No. Observations:                 293   AIC:                            -1284.
Df Residuals:                     283   BIC:                            -1247.
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                                                     coef    std err          t      P>|t|      [0.025      0.975]
----

In [93]:
# 1. Create lag of X

X_lagged = X.shift(1).drop(columns=['const']).rename(columns=lambda x: f"lag_{x}")
const = X['const'].shift(1)
X_lagged = pd.concat([const, X_lagged], axis=1).dropna()

model_2 = sm.OLS(y_current, X_lagged).fit()
print(model_2.summary())



                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.393
Model:                            OLS   Adj. R-squared:                  0.374
Method:                 Least Squares   F-statistic:                     20.36
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           2.50e-26
Time:                        13:03:09   Log-Likelihood:                 421.67
No. Observations:                 293   AIC:                            -823.3
Df Residuals:                     283   BIC:                            -786.5
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                                                         coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------

In [94]:

ols_model, X_selected = backward_elimination(y_current, X_lagged, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())



Dropping 'lag_Natural gas, Europe ($/mmbtu) NGAS_EUR' (p-value = 0.7746)
Dropping 'lag_Coal, Australian ($/mt) COAL_AUS' (p-value = 0.6042)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.392
Model:                            OLS   Adj. R-squared:                  0.377
Method:                 Least Squares   F-statistic:                     26.28
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           1.11e-27
Time:                        13:03:09   Log-Likelihood:                 421.49
No. Observations:                 293   AIC:                            -827.0
Df Residuals:                     285   BIC:                            -797.5
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                                                

In [95]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")
X_lagged = X.shift(1).drop(columns=['const']).rename(columns=lambda x: f"lag_{x}")


# 2. Combine everything
data = pd.concat([y, y_lagged, X_lagged], axis=1).dropna()


# 3. Current y
y_current = data[y.name]   # dependent variable


# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X_lagged.columns]], axis=1))


# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.874
Model:                            OLS   Adj. R-squared:                  0.870
Method:                 Least Squares   F-statistic:                     196.1
Date:                Tue, 21 Oct 2025   Prob (F-statistic):          1.08e-120
Time:                        13:03:09   Log-Likelihood:                 652.31
No. Observations:                 293   AIC:                            -1283.
Df Residuals:                     282   BIC:                            -1242.
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                                                         coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------

In [96]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'lag_Crude oil, Dubai ($/bbl) CRUDE_DUBAI' (p-value = 0.7266)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.874
Model:                            OLS   Adj. R-squared:                  0.870
Method:                 Least Squares   F-statistic:                     218.6
Date:                Tue, 21 Oct 2025   Prob (F-statistic):          7.53e-122
Time:                        13:03:09   Log-Likelihood:                 652.24
No. Observations:                 293   AIC:                            -1284.
Df Residuals:                     283   BIC:                            -1248.
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                                                         coef    std err          t      P>|t|      [0.025      0.975

# UEMOA

In [97]:
# Upload the excel file and save 
x ="/Users/harold/DataAnalyctisandScience/Cointegration_Johansen_test/dataTransportCPI/UEMOA_completes"
df = pd.read_excel(x+".xlsx")


In [98]:

# Define X and y
X = df.drop(columns=["transportation CPI", "Months"])   
X = sm.add_constant(X)       
y = df["transportation CPI"]
ols_model, X_selected = backward_elimination(y, X, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())




Final model summary:
                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.348
Model:                            OLS   Adj. R-squared:                  0.327
Method:                 Least Squares   F-statistic:                     16.81
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           3.70e-22
Time:                        13:03:09   Log-Likelihood:                 585.83
No. Observations:                 294   AIC:                            -1152.
Df Residuals:                     284   BIC:                            -1115.
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                                                     coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------

In [99]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")

# 2. Combine everything
data = pd.concat([y, y_lagged, X], axis=1).dropna()


# 3. Current y
y_current = data[y.name]

# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X.columns]], axis=1))

# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.882
Model:                            OLS   Adj. R-squared:                  0.878
Method:                 Least Squares   F-statistic:                     210.6
Date:                Tue, 21 Oct 2025   Prob (F-statistic):          1.59e-124
Time:                        13:03:09   Log-Likelihood:                 836.63
No. Observations:                 293   AIC:                            -1651.
Df Residuals:                     282   BIC:                            -1611.
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                                                     coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------

In [100]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'Crude oil, Brent ($/bbl) CRUDE_BRENT' (p-value = 0.2782)
Dropping 'Crude oil, Dubai ($/bbl) CRUDE_DUBAI' (p-value = 0.5161)
Dropping 'Crude oil, WTI ($/bbl) CRUDE_WTI' (p-value = 0.6880)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.881
Model:                            OLS   Adj. R-squared:                  0.878
Method:                 Least Squares   F-statistic:                     302.0
Date:                Tue, 21 Oct 2025   Prob (F-statistic):          8.05e-128
Time:                        13:03:09   Log-Likelihood:                 835.72
No. Observations:                 293   AIC:                            -1655.
Df Residuals:                     285   BIC:                            -1626.
Df Model:                           7                                         
Covariance Type:            nonrobust                                 

In [101]:
# 1. Create lag of X

X_lagged = X.shift(1).drop(columns=['const']).rename(columns=lambda x: f"lag_{x}")
const = X['const'].shift(1)
X_lagged = pd.concat([const, X_lagged], axis=1).dropna()

model_2 = sm.OLS(y_current, X_lagged).fit()
print(model_2.summary())



                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.389
Model:                            OLS   Adj. R-squared:                  0.370
Method:                 Least Squares   F-statistic:                     20.05
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           5.65e-26
Time:                        13:03:09   Log-Likelihood:                 595.91
No. Observations:                 293   AIC:                            -1172.
Df Residuals:                     283   BIC:                            -1135.
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                                                         coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------

In [102]:

ols_model, X_selected = backward_elimination(y_current, X_lagged, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())



Dropping 'lag_Crude oil, WTI ($/bbl) CRUDE_WTI' (p-value = 0.1839)
Dropping 'lag_Crude oil, Dubai ($/bbl) CRUDE_DUBAI' (p-value = 0.4362)
Dropping 'lag_Crude oil, average ($/bbl) CRUDE_PETRO' (p-value = 0.1048)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.379
Model:                            OLS   Adj. R-squared:                  0.366
Method:                 Least Squares   F-statistic:                     29.04
Date:                Tue, 21 Oct 2025   Prob (F-statistic):           4.38e-27
Time:                        13:03:10   Log-Likelihood:                 593.33
No. Observations:                 293   AIC:                            -1173.
Df Residuals:                     286   BIC:                            -1147.
Df Model:                           6                                         
Covariance Type:            nonrobust                   

In [103]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")
X_lagged = X.shift(1).drop(columns=['const']).rename(columns=lambda x: f"lag_{x}")


# 2. Combine everything
data = pd.concat([y, y_lagged, X_lagged], axis=1).dropna()


# 3. Current y
y_current = data[y.name]   # dependent variable


# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X_lagged.columns]], axis=1))


# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.882
Model:                            OLS   Adj. R-squared:                  0.878
Method:                 Least Squares   F-statistic:                     211.2
Date:                Tue, 21 Oct 2025   Prob (F-statistic):          1.14e-124
Time:                        13:03:10   Log-Likelihood:                 836.98
No. Observations:                 293   AIC:                            -1652.
Df Residuals:                     282   BIC:                            -1611.
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                                                         coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------

In [104]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'lag_Natural gas, Europe ($/mmbtu) NGAS_EUR' (p-value = 0.6057)
Dropping 'lag_Liquefied natural gas, Japan ($/mmbtu) NGAS_JP' (p-value = 0.6630)
Dropping 'lag_Natural gas, US ($/mmbtu) NGAS_US' (p-value = 0.4600)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:     transportation CPI   R-squared:                       0.882
Model:                            OLS   Adj. R-squared:                  0.879
Method:                 Least Squares   F-statistic:                     303.7
Date:                Tue, 21 Oct 2025   Prob (F-statistic):          3.90e-128
Time:                        13:03:10   Log-Likelihood:                 836.46
No. Observations:                 293   AIC:                            -1657.
Df Residuals:                     285   BIC:                            -1627.
Df Model:                           7                                         
Covariance Type:            nonrobust        